In [ ]:
# Source Code 7 
# Script to expand the original boundaries and group overlapping observations.

import csv # To read from and write to CSV files
import math # To perform mathematical operations and access math constants
import os # To interact with the operating system (e.g., file paths, directories)
import pandas as pd # For data manipulation and analysis

# Load the CSV file into a DataFrame
input_csv_file = 'mhc-inspected-duplicate-cleaned.csv'

# Specify the file path where you want to save the CSV file
output_csv_file = os.path.join(folder_path, 'mhc-inspected-duplicate-cleaned-grouped.csv')
df = pd.read_csv(input_csv_file, sep=',')

# Calculate the new bounding box boundaries
df['width_deg'] = df['east'] - df['west']
df['height_deg'] = df['north'] - df['south']
df['north_w'] = df['north'] + (3 * df['height_deg'])
df['south_w'] = df['south'] - (3 * df['height_deg'])
df['east_w'] = df['east'] + (3 * df['width_deg'])
df['west_w'] = df['west'] - (3 * df['width_deg'])

# Function to check overlap between two bounding boxes
def check_overlap(bbox1, bbox2):
    return not (bbox1['west_w'] > bbox2['east_w'] or
                bbox1['east_w'] < bbox2['west_w'] or
                bbox1['north_w'] < bbox2['south_w'] or
                bbox1['south_w'] > bbox2['north_w'])

# Create a new column to group overlapping observations
df['group_id'] = None
group_counter = 0

# Iterate through each observation
for i in range(len(df)):
    # If the observation is not yet grouped
    if df.at[i, 'group_id'] is None:
        # Create a group for this observation
        df.at[i, 'group_id'] = group_counter
        group_counter += 1
        
    # Check for overlap with other observations
    for j in range(i+1, len(df)):
        if check_overlap(df.loc[i], df.loc[j]):
            # If there is an overlap, assign the same group ID
            if df.at[j, 'group_id'] is None:
                df.at[j, 'group_id'] = df.at[i, 'group_id']

# Save the DataFrame to a CSV file
df.to_csv(output_csv_file, index=False)

print("DataFrame saved to", output_csv_file)